In [5]:

#!/usr/bin/env python3
# -*- coding: utf-8 -*-

from __future__ import annotations

import argparse
import importlib
import json
import sys
import time
from pathlib import Path
from typing import Any

import undetected_chromedriver as uc


# ==================================================
# Notebookテスト用店舗
# ==================================================
#
# Notebookで実行する場合は、ここだけ変更する。
#
# .pyで実行する場合は、
# --siteで指定した店舗が優先される。
#
# 例:
# python cookie_save.py --site iwakuni_tekisasu_s
#

NOTEBOOK_SITE = "marina_s"


# ==================================================
# ブラウザ設定
# ==================================================

PAGE_WAIT_SECONDS = 30

WINDOW_WIDTH = 600
WINDOW_HEIGHT = 1000

# NoneならChromeバージョンを自動判定
CHROME_VERSION_MAIN: int | None = 139


# ==================================================
# プロジェクトルート検出
# ==================================================

def find_project_root(
    start_path: Path,
) -> Path:
    """
    config/ と utils/ がある場所を
    プロジェクトルートとして返す。
    """
    current = start_path.resolve()

    if current.is_file():
        current = current.parent

    for candidate in (
        current,
        *current.parents,
    ):
        if (
            (candidate / "config").is_dir()
            and (candidate / "utils").is_dir()
        ):
            return candidate

    raise RuntimeError(
        "PROJECT_ROOTを特定できませんでした。"
        f" 開始位置: {start_path}"
    )


if "__file__" in globals():
    PROJECT_ROOT = find_project_root(
        Path(__file__)
    )
else:
    PROJECT_ROOT = find_project_root(
        Path.cwd()
    )


if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(
        0,
        str(PROJECT_ROOT),
    )


print(
    f"[INFO] PROJECT_ROOT: "
    f"{PROJECT_ROOT}"
)

print(
    f"[INFO] config存在: "
    f"{(PROJECT_ROOT / 'config').is_dir()}"
)

print(
    f"[INFO] utils存在: "
    f"{(PROJECT_ROOT / 'utils').is_dir()}"
)


# ==================================================
# 共通設定
# ==================================================

from config.common import (
    CREDENTIALS_DIR,
    DEFAULT_SITE,
    HR01_INTERFACE,
    SQUID_PROXY,
)

from utils.network_utils import (
    get_global_ip,
    get_squid_ip,
)


# ==================================================
# 店舗選択
# ==================================================

def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(
        description=(
            "店舗設定のCOOKIE_URLへアクセスし、"
            "Cookieを保存します。"
        )
    )

    parser.add_argument(
        "--site",
        default=DEFAULT_SITE,
        help="configフォルダ内の店舗設定名",
    )

    return parser.parse_args()


if "__file__" in globals():
    # .py実行時は--siteを使用
    args = parse_args()

else:
    # Notebook実行時は上部のNOTEBOOK_SITEを使用
    args = argparse.Namespace(
        site=NOTEBOOK_SITE,
    )


site_name = str(
    args.site
).strip()


if not site_name:
    raise ValueError(
        "店舗設定名が空です。"
    )


# ==================================================
# 店舗設定読み込み
# ==================================================

config_file = (
    PROJECT_ROOT
    / "config"
    / f"{site_name}.py"
)


if not config_file.is_file():
    raise FileNotFoundError(
        f"店舗設定が見つかりません: "
        f"{config_file}"
    )


try:
    site_config = importlib.import_module(
        f"config.{site_name}"
    )

except ModuleNotFoundError as exc:
    raise SystemExit(
        "[ERROR] 店舗設定の読み込みに失敗しました: "
        f"config/{site_name}.py"
    ) from exc


# ==================================================
# 必須設定確認
# ==================================================
#
# COOKIE_FILEは不要。
#
# Cookie保存先はこのスクリプト側で
#
# credentials/<SITE_KEY>/cookies.json
#
# として自動生成する。
# ==================================================

required_settings = (
    "SITE_KEY",
    "COOKIE_URL",
)


for setting_name in required_settings:
    if not hasattr(
        site_config,
        setting_name,
    ):
        raise AttributeError(
            f"config/{site_name}.py に "
            f"{setting_name} が設定されていません。"
        )


# ==================================================
# 店舗別設定
# ==================================================

site_key = str(
    site_config.SITE_KEY
).strip()


if not site_key:
    raise ValueError(
        "SITE_KEYが空です。"
    )


shop_name = str(
    getattr(
        site_config,
        "SHOP_NAME",
        getattr(
            site_config,
            "GSHEET_NAME",
            site_name,
        ),
    )
).strip()


cookie_url = str(
    site_config.COOKIE_URL
).strip()


if not cookie_url.startswith(
    (
        "http://",
        "https://",
    )
):
    raise ValueError(
        f"COOKIE_URLが正しいURLではありません: "
        f"{cookie_url}"
    )


# ==================================================
# Cookie保存先
# ==================================================
#
# config側のCOOKIE_FILEは使用しない。
#
# 必ず
#
# credentials/<SITE_KEY>/cookies.json
#
# に保存する。
# ==================================================

cookie_file = (
    CREDENTIALS_DIR
    / site_key
    / "cookies.json"
)


cookie_file.parent.mkdir(
    parents=True,
    exist_ok=True,
)


print(
    f"[INFO] 対象店舗設定: "
    f"{site_name}"
)

print(
    f"[INFO] SITE_KEY: "
    f"{site_key}"
)

print(
    f"[INFO] 店舗名: "
    f"{shop_name}"
)

print(
    f"[INFO] Cookie取得URL: "
    f"{cookie_url}"
)

print(
    f"[INFO] Cookie保存先: "
    f"{cookie_file}"
)

print(
    f"[INFO] 使用Squid: "
    f"{SQUID_PROXY}"
)


# ==================================================
# Cookie保存
# ==================================================

def save_cookies(
    browser: Any,
    destination: Path,
) -> None:
    """
    現在のブラウザのCookieをJSONへ保存する。

    既存Cookieを読み込む処理ではない。

    COOKIE_URLへアクセスした結果、
    ブラウザへ発行されたCookieを保存する。
    """
    cookies = browser.get_cookies()

    with destination.open(
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            cookies,
            file,
            ensure_ascii=False,
            indent=2,
        )

    print(
        f"[COOKIE] 取得件数: "
        f"{len(cookies)}件"
    )

    print(
        f"✅ Cookie保存完了: "
        f"{destination}"
    )


# ==================================================
# ブラウザ起動
# ==================================================

def open_browser() -> Any:
    """
    undetected_chromedriverでChromeを起動する。

    ChromeのHTTP/HTTPS通信は
    SQUID_PROXYを経由させる。
    """
    chrome_options = uc.ChromeOptions()

    # ==============================================
    # Squid Proxy
    # ==============================================

    squid_proxy = str(
        SQUID_PROXY
    ).strip()

    if not squid_proxy:
        raise RuntimeError(
            "SQUID_PROXYが空です。"
        )

    if squid_proxy.startswith(
        (
            "http://",
            "https://",
            "socks4://",
            "socks5://",
        )
    ):
        proxy_server = squid_proxy

    else:
        proxy_server = (
            f"http://{squid_proxy}"
        )

    chrome_options.add_argument(
        f"--proxy-server={proxy_server}"
    )

    # localhost系はProxyを通さない
    chrome_options.add_argument(
        "--proxy-bypass-list="
        "localhost;127.0.0.1;::1"
    )

    print(
        "[BROWSER] Chrome起動開始"
    )

    print(
        f"[BROWSER] Squid Proxy: "
        f"{proxy_server}"
    )

    chrome_arguments: dict[str, Any] = {
        "options": chrome_options,
    }

    if CHROME_VERSION_MAIN is not None:
        chrome_arguments[
            "version_main"
        ] = CHROME_VERSION_MAIN

        print(
            "[BROWSER] Chromeバージョン指定: "
            f"{CHROME_VERSION_MAIN}"
        )

    else:
        print(
            "[BROWSER] Chromeバージョン: "
            "自動判定"
        )

    browser = uc.Chrome(
        **chrome_arguments
    )

    browser.set_window_size(
        WINDOW_WIDTH,
        WINDOW_HEIGHT,
    )

    print(
        "[BROWSER] Chrome起動完了"
    )

    return browser


# ==================================================
# サイトアクセス前のネットワーク確認
# ==================================================

def verify_network_before_access() -> tuple[str, str]:
    """
    COOKIE_URLへアクセスする前に、

    1. HR01_INTERFACEから見えるグローバルIP
    2. SQUID_PROXY経由で見えるグローバルIP

    を取得する。

    HR01とSquidのグローバルIPは、
    モバイル回線側の出口IPの違いによって
    完全一致しない場合があるため、
    IPの完全一致判定は行わない。

    両方のIPを正常に取得できれば
    ネットワーク確認OKとして処理を続行する。
    """

    print(
        "[NET] サイトアクセス前の"
        "ネットワーク確認開始"
    )

    # ----------------------------------------------
    # HR01側IP
    # ----------------------------------------------

    print(
        "[NET] HR01グローバルIP取得開始"
    )

    hr01_ip = get_global_ip(
        HR01_INTERFACE,
        timeout=20,
    )

    print(
        f"[NET] HR01グローバルIP: "
        f"{hr01_ip}"
    )

    # ----------------------------------------------
    # Squid経由IP
    # ----------------------------------------------

    print(
        "[NET] Squid経由グローバルIP取得開始"
    )

    squid_ip = get_squid_ip(
        SQUID_PROXY,
        timeout=20,
    )

    print(
        f"[NET] Squid経由グローバルIP: "
        f"{squid_ip}"
    )

    # ----------------------------------------------
    # 空チェック
    # ----------------------------------------------

    if not hr01_ip:
        raise RuntimeError(
            "HR01グローバルIPを"
            "取得できませんでした。"
        )

    if not squid_ip:
        raise RuntimeError(
            "Squid経由グローバルIPを"
            "取得できませんでした。"
        )

    # ----------------------------------------------
    # IP情報表示
    #
    # HR01とSquidの完全一致判定は行わない。
    # モバイル回線では出口IPが異なる場合がある。
    # ----------------------------------------------

    if hr01_ip == squid_ip:
        print(
            "[NET] HR01 / Squid "
            "グローバルIP一致"
        )

    else:
        print(
            "[NET] HR01 / Squid "
            "グローバルIPは異なります。"
        )

        print(
            f"[NET] HR01: "
            f"{hr01_ip}"
        )

        print(
            f"[NET] Squid: "
            f"{squid_ip}"
        )

        print(
            "[NET] 完全一致判定は行わないため、"
            "処理を続行します。"
        )

    print(
        "[NET] サイトアクセス前の"
        "ネットワーク確認OK"
    )

    return (
        hr01_ip,
        squid_ip,
    )


# ==================================================
# メイン処理
# ==================================================

def main() -> None:
    start_time = time.time()
    browser = None

    try:
        # ==============================================
        # Chrome起動
        # ==============================================

        browser = open_browser()

        # ==============================================
        # COOKIE_URLへアクセスする前に
        # HR01 / SquidのIPを確認
        # ==============================================

        hr01_ip, squid_ip = (
            verify_network_before_access()
        )

        print(
            "[COOKIE] 今回のCookie取得条件"
        )

        print(
            f"[COOKIE] HR01グローバルIP: "
            f"{hr01_ip}"
        )

        print(
            f"[COOKIE] Squid経由グローバルIP: "
            f"{squid_ip}"
        )

        print(
            f"[COOKIE] 保存先: "
            f"{cookie_file}"
        )

        # ==============================================
        # 対象サイトへアクセス
        #
        # 重要:
        # ここまで対象サイトへはアクセスしていない。
        #
        # また、既存cookies.jsonを
        # Chromeへ読み込む処理も行っていない。
        # ==============================================

        print(
            f"[NAV] Cookie取得URLへアクセス: "
            f"{cookie_url}"
        )

        browser.get(
            cookie_url
        )

        # ==============================================
        # Cookie発行待ち
        # ==============================================

        print(
            f"[WAIT] ページ描画・"
            f"Cookie発行待機: "
            f"{PAGE_WAIT_SECONDS}秒"
        )

        time.sleep(
            PAGE_WAIT_SECONDS
        )

        # ==============================================
        # 現在ページ確認
        # ==============================================

        print(
            f"[NAV] 現在URL: "
            f"{browser.current_url}"
        )

        print(
            f"[NAV] ページタイトル: "
            f"{browser.title}"
        )

        # ==============================================
        # Cookie取得・保存
        # ==============================================

        print(
            "[COOKIE] Cookie取得開始"
        )

        save_cookies(
            browser,
            cookie_file,
        )

    finally:
        # ==============================================
        # Chrome終了
        # ==============================================

        if browser is not None:
            print(
                "[BROWSER] browser.quit() 開始"
            )

            try:
                browser.quit()

                print(
                    "[BROWSER] browser.quit() 完了"
                )

            except Exception as quit_error:
                print(
                    "[WARN] ブラウザ終了失敗: "
                    f"{type(quit_error).__name__}: "
                    f"{quit_error}"
                )

        # ==============================================
        # 実行時間
        # ==============================================

        elapsed_time = (
            time.time()
            - start_time
        )

        print(
            f"[TIME] 実行時間: "
            f"{elapsed_time:.2f}秒"
        )


# ==================================================
# 実行
# ==================================================

if __name__ == "__main__":
    main()


[INFO] PROJECT_ROOT: /home/ubuntu/detaslot
[INFO] config存在: True
[INFO] utils存在: True
[INFO] 対象店舗設定: marina_s
[INFO] SITE_KEY: marina_s
[INFO] 店舗名: マリナプローバs
[INFO] Cookie取得URL: https://www.pscube.jp/dedamajyoho-P-townDMMpachi/c760306
[INFO] Cookie保存先: /home/ubuntu/detaslot/credentials/marina_s/cookies.json
[INFO] 使用Squid: http://127.0.0.1:3128
[BROWSER] Chrome起動開始
[BROWSER] Squid Proxy: http://127.0.0.1:3128
[BROWSER] Chromeバージョン指定: 139
[BROWSER] Chrome起動完了
[NET] サイトアクセス前のネットワーク確認開始
[NET] HR01グローバルIP取得開始
[NET] HR01グローバルIP: 211.7.99.170
[NET] Squid経由グローバルIP取得開始
[NET] Squid経由グローバルIP: 211.7.97.104
[NET ERROR] HR01 / Squid IP不一致
[NET ERROR] HR01: 211.7.99.170
[NET ERROR] Squid: 211.7.97.104
[BROWSER] browser.quit() 開始
[BROWSER] browser.quit() 完了
[TIME] 実行時間: 3.72秒


RuntimeError: SquidがHR01経由ではありません。 HR01=211.7.99.170, Squid=211.7.97.104